# LM2011 Sample Post-Refinitiv Runner

Colab wrapper for `lm2011_sample_post_refinitiv_runner.py`. The default paths map the local runner's sample/upstream concepts onto the Drive full-data layout under `/content/drive/MyDrive/Data_LM`.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_ENV_VAR = "NLP_THESIS_REPO_ROOT"
GIT_URL_ENV_VAR = "NLP_THESIS_GIT_URL"
GIT_REF_ENV_VAR = "NLP_THESIS_GIT_REF"
DEFAULT_GIT_URL = "https://github.com/Erew42/NLP_Thesis.git"
DEFAULT_GIT_REF = "main"
CLONE_ROOT = Path("/content/NLP_Thesis")


def _find_repo_root() -> Path | None:
    candidates: list[Path] = []
    env_root = os.environ.get(REPO_ENV_VAR)
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    if IN_COLAB:
        candidates.extend(
            [
                CLONE_ROOT,
                Path("/content/drive/MyDrive/NLP_Thesis"),
                Path("/content/drive/My Drive/NLP_Thesis"),
            ]
        )

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "src" / "thesis_pkg" / "pipeline.py").exists():
            return candidate
    return None


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

repo_root = _find_repo_root()
if repo_root is None and IN_COLAB:
    git_url = os.environ.get(GIT_URL_ENV_VAR, DEFAULT_GIT_URL)
    if CLONE_ROOT.exists() and not (CLONE_ROOT / "src" / "thesis_pkg" / "pipeline.py").exists():
        raise FileExistsError(
            f"{CLONE_ROOT} exists but is not the NLP_Thesis repo. Remove it or set {REPO_ENV_VAR}."
        )
    if not CLONE_ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", git_url, str(CLONE_ROOT)])
    repo_root = CLONE_ROOT

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the NLP_Thesis checkout. Run from the repo root, set NLP_THESIS_REPO_ROOT, or use Colab."
    )

repo_root = repo_root.resolve()
if IN_COLAB and (repo_root / ".git").exists() and os.environ.get("NLP_THESIS_SKIP_GIT_UPDATE", "0") != "1":
    git_ref = os.environ.get(GIT_REF_ENV_VAR, DEFAULT_GIT_REF)
    fetch_code = subprocess.call(["git", "-C", str(repo_root), "fetch", "--depth", "1", "origin", git_ref])
    checkout_target = "FETCH_HEAD" if fetch_code == 0 else git_ref
    subprocess.call(["git", "-C", str(repo_root), "checkout", checkout_target])

os.environ[REPO_ENV_VAR] = str(repo_root)
src = repo_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB and os.environ.get("NLP_THESIS_SKIP_PIP_INSTALL", "0") != "1":
    install_target = f"{repo_root}[benchmark]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--editable", install_target])

print({"IN_COLAB": IN_COLAB, "repo_root": str(repo_root), "src_exists": src.exists()})

In [ ]:
from pathlib import Path


def _resolve_colab_drive_root() -> Path:
    for candidate in (
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
        Path("/content/drive"),
    ):
        if candidate.exists():
            return candidate
    return Path("/content/drive")


def _env_optional_str(name: str) -> str | None:
    value = os.environ.get(name)
    if value is None:
        return None
    stripped = value.strip()
    if stripped.lower() in {"", "none", "null"}:
        return None
    return stripped


def _env_path(name: str, default: Path) -> Path:
    value = _env_optional_str(name)
    return Path(value).expanduser() if value is not None else default


def _env_optional_path(name: str) -> Path | None:
    value = _env_optional_str(name)
    return Path(value).expanduser() if value is not None else None


def _env_str(name: str, default: str) -> str:
    value = _env_optional_str(name)
    return value if value is not None else default


def _env_optional_int(name: str) -> int | None:
    value = _env_optional_str(name)
    return int(value) if value is not None else None


def _env_int(name: str, default: int) -> int:
    value = _env_optional_int(name)
    return value if value is not None else default


def _env_bool(name: str, default: bool) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    lowered = value.strip().lower()
    if lowered in {"1", "true", "yes", "y", "on"}:
        return True
    if lowered in {"0", "false", "no", "n", "off"}:
        return False
    raise ValueError(f"Invalid boolean value for {name}: {value!r}")


def _first_existing_path(*paths: Path) -> Path:
    for path in paths:
        if path.exists():
            return path
    return paths[0]


def _check_paths(paths: dict[str, Path], *, require_all: bool = True) -> None:
    rows = []
    missing = []
    for label, path in paths.items():
        exists = path.exists()
        rows.append({"label": label, "path": str(path), "exists": exists})
        if require_all and not exists:
            missing.append(f"{label}: {path}")
    try:
        import polars as pl

        display(pl.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)
    if missing:
        raise FileNotFoundError("Missing required Drive inputs:\n" + "\n".join(missing))

DRIVE_ROOT = _resolve_colab_drive_root()
DEFAULT_WORK_ROOT = DRIVE_ROOT / "Data_LM"
WORK_ROOT = _env_path("SEC_CCM_WORK_ROOT", DEFAULT_WORK_ROOT)
SEC_CCM_RUN_ROOT = _env_path(
    "SEC_CCM_RUN_ROOT",
    WORK_ROOT / "results" / "sec_ccm_unified_runner",
)
SEC_CCM_OUTPUT_DIR = _env_path(
    "SEC_CCM_OUTPUT_DIR",
    SEC_CCM_RUN_ROOT / "sec_ccm_premerge",
)

# Full-data equivalents of the unified runner LM2011 inputs.
SAMPLE_ROOT = WORK_ROOT
UPSTREAM_RUN_ROOT = SEC_CCM_RUN_ROOT
YEAR_MERGED_DIR = _env_optional_path("SEC_CCM_LM2011_YEAR_MERGED_DIR") or _env_path(
    "SEC_CCM_SEC_YEAR_MERGED_DIR",
    WORK_ROOT / "parquet_data" / "_year_merged",
)
DEFAULT_DAILY_PANEL_PATH = (
    WORK_ROOT / "CRSP_Compustat_data" / "derived_data" / "final_flagged_data_compdesc_added.parquet"
)
DAILY_PANEL_PATH = _env_optional_path("SEC_CCM_LM2011_DAILY_PANEL_PATH")
if DAILY_PANEL_PATH is None:
    DAILY_PANEL_PATH = (
        SEC_CCM_OUTPUT_DIR / "final_flagged_data.parquet"
        if _env_optional_str("SEC_CCM_OUTPUT_DIR") is not None
        else DEFAULT_DAILY_PANEL_PATH
    )
CCM_BASE_DIR = _env_optional_path("SEC_CCM_LM2011_CCM_BASE_DIR") or _env_path(
    "SEC_CCM_CCM_BASE_DIR",
    WORK_ROOT / "CRSP_Compustat_data" / "parquet_data",
)
MATCHED_CLEAN_PATH = _env_optional_path("SEC_CCM_LM2011_MATCHED_CLEAN_PATH") or (
    SEC_CCM_OUTPUT_DIR / "sec_ccm_matched_clean.parquet"
)
SAMPLE_BACKBONE_PATH = _env_optional_path("SEC_CCM_LM2011_SAMPLE_BACKBONE_PATH")
if SAMPLE_BACKBONE_PATH is None:
    candidate = SEC_CCM_OUTPUT_DIR / "lm2011_sample_backbone.parquet"
    SAMPLE_BACKBONE_PATH = candidate if candidate.exists() else None
ITEMS_ANALYSIS_DIR = _env_optional_path("SEC_CCM_LM2011_ITEMS_ANALYSIS_DIR") or _env_path(
    "SEC_CCM_ITEMS_ANALYSIS_DIR",
    SEC_CCM_RUN_ROOT / "items_analysis",
)
OUTPUT_DIR = _env_path(
    "SEC_CCM_LM2011_OUTPUT_DIR",
    WORK_ROOT / "results" / "lm2011_sample_post_refinitiv_runner",
)
DOC_OWNERSHIP_DIR = _env_path(
    "SEC_CCM_REFINITIV_DOC_OWNERSHIP_LM2011_DIR",
    SEC_CCM_RUN_ROOT / "refinitiv_doc_ownership_lm2011",
)
DOC_OWNERSHIP_PATH = DOC_OWNERSHIP_DIR / "refinitiv_lm2011_doc_ownership.parquet"
DOC_ANALYST_DIR = _env_path(
    "SEC_CCM_REFINITIV_DOC_ANALYST_LM2011_DIR",
    SEC_CCM_RUN_ROOT / "refinitiv_doc_analyst_lm2011",
)
DOC_ANALYST_SELECTED_PATH = DOC_ANALYST_DIR / "refinitiv_doc_analyst_selected.parquet"
local_work_base = _env_optional_path("SEC_CCM_LOCAL_WORK")
if local_work_base is None:
    LOCAL_WORK_ROOT = (
        Path("/content/_batch_work") / "lm2011_post_refinitiv"
        if IN_COLAB
        else repo_root / ".tmp" / "lm2011_post_refinitiv"
    )
else:
    LOCAL_WORK_ROOT = local_work_base / "lm2011_post_refinitiv"

# Put these files under Data_LM/LM2011_additional_data, or edit this value to a mounted Drive
# folder containing Fin-Neg.txt, LM2011_MasterDictionary.txt,
# F-F_Research_Data_Factors_daily.csv, F-F_Research_Data_Factors.csv,
# F-F_Momentum_Factor.csv, and FF_Siccodes_48_Industries.txt.
ADDITIONAL_DATA_DIR = _env_optional_path("SEC_CCM_LM2011_ADDITIONAL_DATA_DIR")
if ADDITIONAL_DATA_DIR is None:
    ADDITIONAL_DATA_DIR = _first_existing_path(
        WORK_ROOT / "LM2011_additional_data",
        repo_root / "full_data_run" / "LM2011_additional_data",
    )

LEGACY_TEXT_FEATURE_BATCH_SIZE = _env_optional_int("SEC_CCM_LM2011_TEXT_FEATURE_BATCH_SIZE")
FULL_10K_CLEANING_CONTRACT = _env_str(
    "SEC_CCM_LM2011_FULL_10K_CLEANING_CONTRACT",
    "lm2011_paper",
)  # current | lm2011_paper
FULL_10K_TEXT_FEATURE_BATCH_SIZE = _env_int(
    "SEC_CCM_LM2011_FULL_10K_TEXT_FEATURE_BATCH_SIZE",
    LEGACY_TEXT_FEATURE_BATCH_SIZE if LEGACY_TEXT_FEATURE_BATCH_SIZE is not None else 1000,
)
MDA_TEXT_FEATURE_BATCH_SIZE = _env_int(
    "SEC_CCM_LM2011_MDA_TEXT_FEATURE_BATCH_SIZE",
    LEGACY_TEXT_FEATURE_BATCH_SIZE if LEGACY_TEXT_FEATURE_BATCH_SIZE is not None else 1000,
)
RECOMPUTE_TEXT_FEATURES_FULL_10K = _env_bool(
    "SEC_CCM_LM2011_RECOMPUTE_TEXT_FEATURES_FULL_10K",
    False,
)
RECOMPUTE_TEXT_FEATURES_MDA = _env_bool(
    "SEC_CCM_LM2011_RECOMPUTE_TEXT_FEATURES_MDA",
    False,
)
RECOMPUTE_STRATEGY_TEXT_FEATURES_FULL_10K = _env_bool(
    "SEC_CCM_LM2011_RECOMPUTE_STRATEGY_TEXT_FEATURES_FULL_10K",
    False,
)
RECOMPUTE_EVENT_SCREEN_SURFACE = _env_bool(
    "SEC_CCM_LM2011_RECOMPUTE_EVENT_SCREEN_SURFACE",
    False,
)
RECOMPUTE_EVENT_PANEL = _env_bool(
    "SEC_CCM_LM2011_RECOMPUTE_EVENT_PANEL",
    False,
)
RECOMPUTE_REGRESSION_TABLES = _env_bool(
    "SEC_CCM_LM2011_RECOMPUTE_REGRESSION_TABLES",
    False,
)
# The unified LM2011 extension runner now reads these same recompute flags when it builds
# extension shared prereqs. That affects the shared full-10-K token-count artifact plus
# optional event-screen-surface / event-panel refreshes. MD&A recompute is carried for
# parity with the main LM2011 runner but does not change extension shared prereqs today.
MONTHLY_STOCK_PATH = _env_optional_path("SEC_CCM_LM2011_MONTHLY_STOCK_PATH")
LM2011_STRATEGY_TEXT_FEATURES_FULL_10K_PATH = _env_optional_path(
    "SEC_CCM_LM2011_STRATEGY_TEXT_FEATURES_FULL_10K_PATH"
)
FF_MONTHLY_WITH_MOM_PATH = _env_optional_path("SEC_CCM_LM2011_FF_MONTHLY_WITH_MOM_PATH")
EVENT_WINDOW_DOC_BATCH_SIZE = _env_int("SEC_CCM_LM2011_EVENT_WINDOW_DOC_BATCH_SIZE", 50)
PRINT_RAM_STATS = _env_bool("SEC_CCM_PRINT_RAM_STATS", False)
RAM_LOG_INTERVAL_BATCHES = _env_int("SEC_CCM_RAM_LOG_INTERVAL_BATCHES", 10)

required_paths = {
    "WORK_ROOT": WORK_ROOT,
    "UPSTREAM_RUN_ROOT": UPSTREAM_RUN_ROOT,
    "YEAR_MERGED_DIR": YEAR_MERGED_DIR,
    "DAILY_PANEL_PATH": DAILY_PANEL_PATH,
    "CCM_BASE_DIR": CCM_BASE_DIR,
    "MATCHED_CLEAN_PATH": MATCHED_CLEAN_PATH,
    "ITEMS_ANALYSIS_DIR": ITEMS_ANALYSIS_DIR,
    "ADDITIONAL_DATA_DIR": ADDITIONAL_DATA_DIR,
    "FF_DAILY_CSV": ADDITIONAL_DATA_DIR / "F-F_Research_Data_Factors_daily.csv",
    "FF_MONTHLY_CSV": ADDITIONAL_DATA_DIR / "F-F_Research_Data_Factors.csv",
    "MOMENTUM_MONTHLY_CSV": ADDITIONAL_DATA_DIR / "F-F_Momentum_Factor.csv",
    "FF48_SICCODES": ADDITIONAL_DATA_DIR / "FF_Siccodes_48_Industries.txt",
    "LM_MASTER_DICTIONARY": ADDITIONAL_DATA_DIR / "LM2011_MasterDictionary.txt",
}
_check_paths(required_paths)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_WORK_ROOT.mkdir(parents=True, exist_ok=True)
print({
    "OUTPUT_DIR": str(OUTPUT_DIR),
    "LOCAL_WORK_ROOT": str(LOCAL_WORK_ROOT),
    "DOC_OWNERSHIP_PATH": str(DOC_OWNERSHIP_PATH),
    "DOC_ANALYST_SELECTED_PATH": str(DOC_ANALYST_SELECTED_PATH),
    "FULL_10K_CLEANING_CONTRACT": FULL_10K_CLEANING_CONTRACT,
    "RECOMPUTE_TEXT_FEATURES_FULL_10K": RECOMPUTE_TEXT_FEATURES_FULL_10K,
    "RECOMPUTE_TEXT_FEATURES_MDA": RECOMPUTE_TEXT_FEATURES_MDA,
    "RECOMPUTE_EVENT_SCREEN_SURFACE": RECOMPUTE_EVENT_SCREEN_SURFACE,
    "RECOMPUTE_EVENT_PANEL": RECOMPUTE_EVENT_PANEL,
    "RECOMPUTE_REGRESSION_TABLES": RECOMPUTE_REGRESSION_TABLES,
    "SAMPLE_BACKBONE_PATH": str(SAMPLE_BACKBONE_PATH) if SAMPLE_BACKBONE_PATH is not None else None,
    "BACKBONE_MODE": "reuse_prebuilt" if SAMPLE_BACKBONE_PATH is not None else "metadata_only_rebuild",
    "FULL_10K_TEXT_FEATURE_BATCH_SIZE": FULL_10K_TEXT_FEATURE_BATCH_SIZE,
    "MDA_TEXT_FEATURE_BATCH_SIZE": MDA_TEXT_FEATURE_BATCH_SIZE,
    "EVENT_WINDOW_DOC_BATCH_SIZE": EVENT_WINDOW_DOC_BATCH_SIZE,
    "PRINT_RAM_STATS": PRINT_RAM_STATS,
    "RAM_LOG_INTERVAL_BATCHES": RAM_LOG_INTERVAL_BATCHES,
})


In [ ]:
from thesis_pkg.notebooks_and_scripts import lm2011_sample_post_refinitiv_runner as runner

RUN_ARGS = [
    "--sample-root", str(SAMPLE_ROOT),
    "--upstream-run-root", str(UPSTREAM_RUN_ROOT),
    "--additional-data-dir", str(ADDITIONAL_DATA_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--local-work-root", str(LOCAL_WORK_ROOT),
    "--year-merged-dir", str(YEAR_MERGED_DIR),
    "--matched-clean-path", str(MATCHED_CLEAN_PATH),
    "--daily-panel-path", str(DAILY_PANEL_PATH),
    "--items-analysis-dir", str(ITEMS_ANALYSIS_DIR),
    "--ccm-base-dir", str(CCM_BASE_DIR),
    "--doc-ownership-path", str(DOC_OWNERSHIP_PATH),
    "--doc-analyst-selected-path", str(DOC_ANALYST_SELECTED_PATH),
    "--full-10k-cleaning-contract", FULL_10K_CLEANING_CONTRACT,
    "--full-10k-text-feature-batch-size", str(FULL_10K_TEXT_FEATURE_BATCH_SIZE),
    "--mda-text-feature-batch-size", str(MDA_TEXT_FEATURE_BATCH_SIZE),
    "--event-window-doc-batch-size", str(EVENT_WINDOW_DOC_BATCH_SIZE),
]
if RECOMPUTE_TEXT_FEATURES_FULL_10K:
    RUN_ARGS.append("--recompute-text-features-full-10k")
if RECOMPUTE_TEXT_FEATURES_MDA:
    RUN_ARGS.append("--recompute-text-features-mda")
if RECOMPUTE_STRATEGY_TEXT_FEATURES_FULL_10K:
    RUN_ARGS.append("--recompute-strategy-text-features-full-10k")
if RECOMPUTE_EVENT_SCREEN_SURFACE:
    RUN_ARGS.append("--recompute-event-screen-surface")
if RECOMPUTE_EVENT_PANEL:
    RUN_ARGS.append("--recompute-event-panel")
if RECOMPUTE_REGRESSION_TABLES:
    RUN_ARGS.append("--recompute-regression-tables")
if SAMPLE_BACKBONE_PATH is not None:
    RUN_ARGS.extend(["--sample-backbone-path", str(SAMPLE_BACKBONE_PATH)])
if MONTHLY_STOCK_PATH is not None:
    RUN_ARGS.extend(["--monthly-stock-path", str(MONTHLY_STOCK_PATH)])
if LM2011_STRATEGY_TEXT_FEATURES_FULL_10K_PATH is not None:
    RUN_ARGS.extend([
        "--strategy-text-features-full-10k-path",
        str(LM2011_STRATEGY_TEXT_FEATURES_FULL_10K_PATH),
    ])
if FF_MONTHLY_WITH_MOM_PATH is not None:
    RUN_ARGS.extend(["--ff-monthly-with-mom-path", str(FF_MONTHLY_WITH_MOM_PATH)])
if PRINT_RAM_STATS:
    RUN_ARGS.append("--print-ram-stats")
RUN_ARGS.extend(["--ram-log-interval-batches", str(RAM_LOG_INTERVAL_BATCHES)])

RUN_ARGS


In [ ]:
ARGS = runner.parse_args(RUN_ARGS)
LM2011_STAGE_ENV_NAMES = (
    "SEC_CCM_RUN_LM2011_POST_REFINITIV",
    *[f"SEC_CCM_RUN_LM2011_{stage_name.upper()}" for stage_name in runner.LM2011_ALL_STAGE_NAMES],
)
USE_UNIFIED_STAGE_ENV = any(name in os.environ for name in LM2011_STAGE_ENV_NAMES)

if USE_UNIFIED_STAGE_ENV:
    ENABLED_STAGES = runner.resolve_enabled_lm2011_stage_names_from_env()
    RUN_CFG = runner.build_lm2011_post_refinitiv_run_config(
        ARGS,
        enabled_stages=ENABLED_STAGES,
        fail_closed_for_enabled_stages=True,
    )
else:
    RUN_CFG = runner.build_lm2011_post_refinitiv_run_config(ARGS)
    ENABLED_STAGES = RUN_CFG.enabled_stages

print({
    "use_unified_stage_env": USE_UNIFIED_STAGE_ENV,
    "enabled_stage_count": len(ENABLED_STAGES),
    "enabled_stages": list(ENABLED_STAGES),
    "fail_closed_for_enabled_stages": RUN_CFG.fail_closed_for_enabled_stages,
})
exit_code = runner.run_lm2011_post_refinitiv_pipeline(RUN_CFG)
assert exit_code == 0

In [ ]:
import json
import polars as pl

manifest_path = OUTPUT_DIR / "lm2011_sample_run_manifest.json"
print({"manifest_path": str(manifest_path)})
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps({"run_status": manifest.get("run_status"), "row_counts": manifest.get("row_counts")}, indent=2))

stages = manifest.get("stages", {})
for stage_name, stage in stages.items():
    print(stage_name, stage.get("status"), stage.get("reason") or "")

for key in ("sample_backbone", "event_panel", "table_iv_results", "table_v_results"):
    path_raw = manifest.get("artifacts", {}).get(key)
    if path_raw:
        path = Path(path_raw)
        print(key, pl.scan_parquet(path).select(pl.len().alias("rows")).collect().item())